Q10: Data Wrangling

Data wrangling on real estate prices.

Use RealEstate_Prices.csv from the same folder. This is the complete Housing Prices dataset with 545 rows and 13 columns, matching the supplied practical.

Source: https://www.kaggle.com/datasets/yasserh/housing-prices-dataset

The downloaded data is unchanged and saved as RealEstate_Prices.csv. Cleaned data is saved separately as cleaned_realestate_prices.csv.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("RealEstate_Prices.csv")
print(df.head())
df.info()
print("\nMissing Values:")
print(df.isnull().sum())

      price  area  bedrooms  ...  parking  prefarea furnishingstatus
0  13300000  7420         4  ...        2       yes        furnished
1  12250000  8960         4  ...        3        no        furnished
2  12250000  9960         3  ...        2       yes   semi-furnished
3  12215000  7500         4  ...        3       yes        furnished
4  11410000  7420         4  ...        2        no        furnished

[5 rows x 13 columns]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 545 entries, 0 to 544
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   price             545 non-null    int64 
 1   area              545 non-null    int64 
 2   bedrooms          545 non-null    int64 
 3   bathrooms         545 non-null    int64 
 4   stories           545 non-null    int64 
 5   mainroad          545 non-null    object
 6   guestroom         545 non-null    object
 7   basement          545 non-null    objec

In [2]:
# Clean column names
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(r"[^a-zA-Z0-9_]", "", regex=True)
)

In [3]:
# Remove duplicates, extra spaces and missing values
df = df.drop_duplicates().copy()

for col in df.select_dtypes("object"):
    df[col] = df[col].str.strip()

df = df.dropna()

In [4]:
# Encode categorical columns
categorical = df.select_dtypes("object").columns
df = pd.get_dummies(df, columns=categorical, drop_first=True)

In [5]:
# Remove outliers using IQR
numeric = df.select_dtypes(include=np.number).columns

for col in numeric:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    df = df[
        (df[col] >= q1 - 1.5 * iqr) &
        (df[col] <= q3 + 1.5 * iqr)
    ]

In [6]:
print("Cleaned Data:")
print(df.head())
print("\nAverage values:")
print(df.mean(numeric_only=True))
print("\nCleaned Shape:", df.shape)

Cleaned Data:
      price  ...  furnishingstatus_unfurnished
15  9100000  ...                         False
20  8750000  ...                         False
22  8645000  ...                         False
27  8400000  ...                         False
40  7875000  ...                         False

[5 rows x 14 columns]

Average values:
price                              4.045012e+06
area                               4.623811e+03
bedrooms                           2.742466e+00
bathrooms                          1.000000e+00
stories                            1.553425e+00
parking                            5.369863e-01
mainroad_yes                       8.383562e-01
guestroom_yes                      1.397260e-01
basement_yes                       3.178082e-01
hotwaterheating_yes                3.835616e-02
airconditioning_yes                2.356164e-01
prefarea_yes                       2.082192e-01
furnishingstatus_semi-furnished    4.109589e-01
furnishingstatus_unfurnished       3.780

In [7]:
# Save the cleaned dataset
df.to_csv("cleaned_realestate_prices.csv", index=False)
print("Saved cleaned_realestate_prices.csv")

Saved cleaned_realestate_prices.csv
